In [1]:
import csv
from collections import defaultdict

def extract_entities(filepath):
    entities = defaultdict(list)
    current_tokens, current_type = [], None
    with open(filepath, encoding='utf-8-sig') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_tokens and current_type:
                    entities[current_type].append(" ".join(current_tokens))
                current_tokens, current_type = [], None
                continue
            parts = line.split('\t')
            if len(parts) < 2:
                continue
            token, tag = parts[0], parts[1]
            if tag.startswith("B-"):
                if current_tokens and current_type:
                    entities[current_type].append(" ".join(current_tokens))
                current_tokens = [token]
                current_type = tag[2:]
            elif tag.startswith("I-") and current_tokens:
                current_tokens.append(token)
            else:
                if current_tokens and current_type:
                    entities[current_type].append(" ".join(current_tokens))
                current_tokens, current_type = [], None
    return entities

base = "/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/"

train = extract_entities(base + "train_final.conll")
test  = extract_entities(base + "NER_Irish_test.conll")
val   = extract_entities(base + "NER_Irish_validation.conll")

# Combine unique entities across all splits, tracking which splits each appears in
combined = defaultdict(lambda: {"entity": "", "in_train": False, "in_val": False, "in_test": False, "train_count": 0})

for etype in ["PER", "ORG", "LOC"]:
    for ent in train[etype]:
        combined[etype + "|||" + ent]["entity"] = ent
        combined[etype + "|||" + ent]["in_train"] = True
        combined[etype + "|||" + ent]["train_count"] += 1
    for ent in set(val[etype]):
        combined[etype + "|||" + ent]["entity"] = ent
        combined[etype + "|||" + ent]["in_val"] = True
    for ent in set(test[etype]):
        combined[etype + "|||" + ent]["entity"] = ent
        combined[etype + "|||" + ent]["in_test"] = True

for etype in ["PER", "ORG", "LOC"]:
    rows = [
        {
            "entity": v["entity"],
            "train_count": v["train_count"],
            "in_train": v["in_train"],
            "in_val": v["in_val"],
            "in_test": v["in_test"]
        }
        for k, v in combined.items() if k.startswith(etype + "|||")
    ]
    rows.sort(key=lambda x: -x["train_count"])
    outpath = base + f"{etype.lower()}_entities.csv"
    with open(outpath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["entity", "train_count", "in_train", "in_val", "in_test"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved {len(rows)} unique {etype} entities → {etype.lower()}_entities.csv")

Saved 651 unique PER entities → per_entities.csv
Saved 656 unique ORG entities → org_entities.csv
Saved 573 unique LOC entities → loc_entities.csv


In [2]:
import pandas as pd

base = "/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/"

per = pd.read_csv(base + "per_entities.csv")
org = pd.read_csv(base + "org_entities.csv")
loc = pd.read_csv(base + "loc_entities.csv")

print("=== TOP 20 PER ===")
print(per.head(20).to_string(index=False))
print("\n=== TOP 20 ORG ===")
print(org.head(20).to_string(index=False))
print("\n=== TOP 20 LOC ===")
print(loc.head(20).to_string(index=False))

=== TOP 20 PER ===
                  entity  train_count  in_train  in_val  in_test
              Aire Stáit           10      True    True     True
               Taoiseach            6      True   False     True
                    Aire            5      True   False    False
                   tAire            5      True   False    False
               Humphreys            5      True   False    False
            Éamon Ó Cuív            4      True   False    False
                  Ó Cuív            4      True    True     True
                  Butler            4      True   False    False
   Leas-Cheann Comhairle            3      True   False    False
             tAire Stáit            3      True   False    False
            Bertie Ahern            3      True   False     True
                    Ring            3      True   False    False
          Margaret Hayes            3      True   False    False
                      Dé            2      True   False    False
      

In [6]:
import requests
import pandas as pd
import time

base = "/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/"

per = pd.read_csv(base + "per_entities.csv")
org = pd.read_csv(base + "org_entities.csv")
loc = pd.read_csv(base + "loc_entities.csv")

HEADERS = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)"
}

def search_wikidata(entity_string, entity_type):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "search": entity_string,
        "language": "en",
        "uselang": "en",
        "limit": 1,
        "format": "json"
    }
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=10)
        data = r.json()
        results = data.get("search", [])
        if not results:
            return {"entity": entity_string, "type": entity_type,
                    "found": False, "has_irish_label": False, "qid": None, "description": None}
        top = results[0]
        qid = top.get("id")
        detail_url = f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"
        r2 = requests.get(detail_url, headers=HEADERS, timeout=10)
        detail = r2.json()
        labels = detail.get("entities", {}).get(qid, {}).get("labels", {})
        has_irish = "ga" in labels
        return {
            "entity": entity_string,
            "type": entity_type,
            "found": True,
            "qid": qid,
            "has_irish_label": has_irish,
            "irish_label": labels.get("ga", {}).get("value", None),
            "description": top.get("description", None)
        }
    except Exception as e:
        return {"entity": entity_string, "type": entity_type,
                "found": None, "has_irish_label": None, "qid": None, "description": str(e)}

sample_per = per.head(30)["entity"].tolist()
sample_org = org.head(30)["entity"].tolist()
sample_loc = loc.head(30)["entity"].tolist()

results = []
total = len(sample_per) + len(sample_org) + len(sample_loc)
count = 0

for entity in sample_per:
    results.append(search_wikidata(entity, "PER"))
    count += 1
    print(f"[{count}/{total}] PER: {entity}")
    time.sleep(0.5)

for entity in sample_org:
    results.append(search_wikidata(entity, "ORG"))
    count += 1
    print(f"[{count}/{total}] ORG: {entity}")
    time.sleep(0.5)

for entity in sample_loc:
    results.append(search_wikidata(entity, "LOC"))
    count += 1
    print(f"[{count}/{total}] LOC: {entity}")
    time.sleep(0.5)

df = pd.DataFrame(results)
df.to_csv(base + "wikidata_coverage_check.csv", index=False)

print("\n=== COVERAGE SUMMARY ===")
for etype in ["PER", "ORG", "LOC"]:
    subset = df[df["type"] == etype]
    found = subset["found"].sum()
    irish = subset["has_irish_label"].sum()
    total_checked = len(subset)
    print(f"\n{etype} (top {total_checked}):")
    print(f"  Found in Wikidata:      {found}/{total_checked} ({100*found//total_checked}%)")
    print(f"  Has Irish label:        {irish}/{total_checked} ({100*irish//total_checked}%)")
    print(f"  Not found:              {total_checked - found}/{total_checked}")

print("\n=== NOT FOUND IN WIKIDATA ===")
not_found = df[df["found"] == False]
for _, row in not_found.iterrows():
    print(f"  [{row['type']}] {row['entity']}")

print("\n=== FOUND BUT NO IRISH LABEL ===")
no_irish = df[(df["found"] == True) & (df["has_irish_label"] == False)]
for _, row in no_irish.iterrows():
    print(f"  [{row['type']}] {row['entity']} ({row['qid']}) — {row['description']}")

[1/90] PER: Aire Stáit
[2/90] PER: Taoiseach
[3/90] PER: Aire
[4/90] PER: tAire
[5/90] PER: Humphreys
[6/90] PER: Éamon Ó Cuív
[7/90] PER: Ó Cuív
[8/90] PER: Butler
[9/90] PER: Leas-Cheann Comhairle
[10/90] PER: tAire Stáit
[11/90] PER: Bertie Ahern
[12/90] PER: Ring
[13/90] PER: Margaret Hayes
[14/90] PER: Dé
[15/90] PER: Aire Oideachais
[16/90] PER: Ryan Tubridy
[17/90] PER: Tánaiste
[18/90] PER: gCathaoirleach Gníomhach
[19/90] PER: Aire Airgeadais
[20/90] PER: Ó Snodaigh
[21/90] PER: Noel
[22/90] PER: Aire Turasóireachta , Cultúir , Ealaíon , Gaeltachta , Spóirt agus Meán
[23/90] PER: Uachtarán
[24/90] PER: Concubhar Ó Liatháin
[25/90] PER: Pádraig
[26/90] PER: Piaras Ó Dochartaigh
[27/90] PER: Liam Ó Muirthile
[28/90] PER: Oisín
[29/90] PER: Gráinne
[30/90] PER: Joe Éinniú
[31/90] ORG: Údarás na Gaeltachta
[32/90] ORG: RTÉ
[33/90] ORG: TG4
[34/90] ORG: Comhairle Cathrach Bhaile Átha Cliath
[35/90] ORG: Parlaimint na hEorpa
[36/90] ORG: Fianna Fáil
[37/90] ORG: HSE
[38/90] ORG: Ria

In [7]:
import requests

HEADERS = {"User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)"}

# Test 1: Logainm API
print("=== LOGAINM ===")
r = requests.get("https://www.logainm.ie/api/v1.0/1166137", headers=HEADERS, timeout=10)
print(f"Status: {r.status_code}")
if r.status_code == 200:
    print(r.json())

# Test 2: Oireachtas API
print("\n=== OIREACHTAS ===")
r2 = requests.get("https://api.oireachtas.ie/v1/members?limit=5&format=json", headers=HEADERS, timeout=10)
print(f"Status: {r2.status_code}")
if r2.status_code == 200:
    data = r2.json()
    member = data["results"]["member"][0]["memberCode"]
    print(f"Sample member: {member}")
    print(f"Total members available: {data['pagination']['totalCount']}")

=== LOGAINM ===
Status: 401

=== OIREACHTAS ===
Status: 200


TypeError: list indices must be integers or slices, not str

In [8]:
import json

print("=== LOGAINM STATUS ===")
print(f"Status: {r.status_code}")
if r.status_code == 200:
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))

print("\n=== OIREACHTAS STRUCTURE ===")
print(json.dumps(data, indent=2, ensure_ascii=False)[:3000])

=== LOGAINM STATUS ===
Status: 401

=== OIREACHTAS STRUCTURE ===
{
  "head": {
    "counts": {
      "memberCount": 1928,
      "resultCount": 1928
    }
  },
  "results": [
    {
      "member": {
        "gender": "",
        "uri": "https://data.oireachtas.ie/ie/oireachtas/member/id/Henry-J-J-Abbott.D.1987-03-10",
        "pId": "HenryJJAbbott",
        "firstName": "Henry J. J.",
        "lastName": "Abbott",
        "image": false,
        "wikiTitle": null,
        "memberships": [
          {
            "membership": {
              "house": {
                "uri": "https://data.oireachtas.ie/ie/oireachtas/house/dail/25",
                "houseCode": "dail",
                "houseNo": "25",
                "chamberType": "house",
                "showAs": "25th Dáil"
              },
              "represents": [
                {
                  "represent": {
                    "representCode": "Longford-Westmeath",
                    "uri": "https://data.oireachtas.ie/i

In [9]:
LOGAINM_API_KEY = "JNOAg0KLpzdBvml9VpIWfBg5nlHUrE"

HEADERS_LOGAINM = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

In [10]:
import requests

r = requests.get(
    "https://www.logainm.ie/api/v1.0/1166137",
    headers=HEADERS_LOGAINM,
    timeout=10
)
print(f"Status: {r.status_code}")
import json
print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:3000])

Status: 200
{
  "id": 1166137,
  "dateCreated": "2008-03-14T15:32:29.497",
  "dateModified": "2021-07-29T11:55:31.113",
  "permalink": "https://www.logainm.ie/1166137.aspx",
  "featured": [],
  "cluster": {
    "focusID": 1166137,
    "members": [
      {
        "placeID": 1166137,
        "category": {
          "id": "B",
          "nameEN": "town",
          "nameGA": "baile"
        }
      },
      {
        "placeID": 1372978,
        "category": {
          "id": "TR",
          "nameEN": "electoral division",
          "nameGA": "toghroinn"
        }
      }
    ]
  },
  "placenames": [
    {
      "id": 913984,
      "language": "ga",
      "wording": "Na Forbacha",
      "genitive": "na bhForbach",
      "main": true,
      "audio": {
        "fileName": "1166137g.mp3",
        "uri": "https://fionstorage.blob.core.windows.net/topoaudio/1166137g.mp3"
      }
    },
    {
      "id": 913983,
      "language": "en",
      "wording": "Furbogh",
      "main": true,
      "audio"

In [12]:
import requests
import pandas as pd
import json
import time

LOGAINM_API_KEY = "JNOAg0KLpzdBvml9VpIWfBg5nlHUrE"
HEADERS_LOGAINM = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

base = "/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/"
loc = pd.read_csv(base + "loc_entities.csv")

def query_logainm(place_string):
    """Search Logainm for a place string and return canonical forms and hierarchy."""
    url = "https://www.logainm.ie/api/v1.0/"
    params = {"Query": place_string, "PerPage": 1}
    try:
        r = requests.get(url, headers=HEADERS_LOGAINM, params=params, timeout=10)
        if r.status_code != 200:
            return {"entity": place_string, "found": False, "logainm_id": None,
                    "name_ga": None, "name_en": None, "genitive": None,
                    "county": None, "gaeltacht": False, "error": r.status_code}
        data = r.json()
        results = data.get("results", [])
        if not results:
            return {"entity": place_string, "found": False, "logainm_id": None,
                    "name_ga": None, "name_en": None, "genitive": None,
                    "county": None, "gaeltacht": False, "error": None}
        top = results[0]
        # Extract Irish and English names
        name_ga, name_en, genitive = None, None, None
        for pn in top.get("placenames", []):
            if pn["language"] == "ga" and pn["main"]:
                name_ga = pn["wording"]
                genitive = pn.get("genitive")
            if pn["language"] == "en" and pn["main"]:
                name_en = pn["wording"]
        # Extract county from hierarchy
        county = None
        for parent in top.get("includedIn", []):
            if parent.get("category", {}).get("id") == "CON":
                county = parent.get("nameGA")
        gaeltacht = top.get("gaeltacht") is not None
        return {
            "entity": place_string,
            "found": True,
            "logainm_id": top.get("id"),
            "name_ga": name_ga,
            "name_en": name_en,
            "genitive": genitive,
            "county": county,
            "gaeltacht": gaeltacht,
            "error": None
        }
    except Exception as e:
        return {"entity": place_string, "found": False, "logainm_id": None,
                "name_ga": None, "name_en": None, "genitive": None,
                "county": None, "gaeltacht": False, "error": str(e)}

results = []
total = len(loc)

for i, row in loc.iterrows():
    entity = row["entity"]
    result = query_logainm(entity)
    results.append(result)
    status = "✓" if result["found"] else "✗"
    print(f"[{i+1}/{total}] {status} {entity}")
    time.sleep(0.3)

df_logainm = pd.DataFrame(results)
df_logainm.to_csv(base + "logainm_coverage.csv", index=False)

found = df_logainm["found"].sum()
gaeltacht = df_logainm[df_logainm["found"] == True]["gaeltacht"].sum()

print(f"\n=== LOGAINM COVERAGE SUMMARY ===")
print(f"Total LOC entities:        {total}")
print(f"Found in Logainm:          {found}/{total} ({100*found//total}%)")
print(f"Of those, in Gaeltacht:    {gaeltacht}/{found}")
print(f"\n=== SAMPLE MATCHES WITH GENITIVE FORMS ===")
sample = df_logainm[df_logainm["genitive"].notna()].head(15)
for _, row in sample.iterrows():
    print(f"  {row['entity']} → canonical: {row['name_ga']} | genitive: {row['genitive']} | county: {row['county']}")

[1/573] ✗ Ghaeltacht
[2/573] ✗ Bhaile Átha Cliath
[3/573] ✗ Éirinn


KeyboardInterrupt: 

In [13]:
import requests
import json

LOGAINM_API_KEY = "JNOAg0KLpzdBvml9VpIWfBg5nlHUrE"
HEADERS_LOGAINM = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

r = requests.get(
    "https://www.logainm.ie/api/v1.0/",
    headers=HEADERS_LOGAINM,
    params={"Query": "Gaillimh", "PerPage": 1},
    timeout=10
)
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2, ensure_ascii=False)[:3000])

ReadTimeout: HTTPSConnectionPool(host='www.logainm.ie', port=443): Read timed out. (read timeout=10)

In [14]:
import requests
import time


HEADERS_LOGAINM = {
    "User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)",
    "X-Api-Key": LOGAINM_API_KEY
}

# Test a few known Logainm IDs from Wikidata
test_ids = [
    (1166137, "Na Forbacha"),
    (25942, "Gaillimh / Galway"),
    (1411548, "Conamara"),
    (26783, "Gaoth Dobhair"),
]

for logainm_id, label in test_ids:
    r = requests.get(
        f"https://www.logainm.ie/api/v1.0/{logainm_id}",
        headers=HEADERS_LOGAINM,
        timeout=10
    )
    print(f"{label}: {r.status_code} — {r.elapsed.total_seconds():.2f}s")
    time.sleep(0.3)

Na Forbacha: 200 — 0.50s
Gaillimh / Galway: 200 — 0.41s
Conamara: 200 — 0.62s
Gaoth Dobhair: 200 — 0.40s


In [15]:
import requests
import pandas as pd
import time

HEADERS = {"User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)"}
base = "/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/"

all_members = []
limit = 100
offset = 0
total = 1928

while offset < total:
    r = requests.get(
        "https://api.oireachtas.ie/v1/members",
        params={"limit": limit, "offset": offset, "format": "json"},
        headers=HEADERS,
        timeout=15
    )
    data = r.json()
    batch = data["results"]
    for item in batch:
        m = item["member"]
        # Extract party and constituency from most recent membership
        party, constituency, house_no = None, None, None
        if m["memberships"]:
            latest = m["memberships"][0]["membership"]
            house_no = latest["house"]["houseNo"]
            if latest["parties"]:
                party = latest["parties"][0]["party"]["showAs"]
            if latest["represents"]:
                constituency = latest["represents"][0]["represent"]["showAs"]
        all_members.append({
            "member_code": m["memberCode"],
            "full_name_en": m["fullName"],
            "first_name": m["firstName"],
            "last_name": m["lastName"],
            "party": party,
            "constituency": constituency,
            "house_no": house_no,
            "uri": m["uri"]
        })
    offset += limit
    print(f"Fetched {min(offset, total)}/{total} members")
    time.sleep(0.3)

df_members = pd.DataFrame(all_members)
df_members.to_csv(base + "oireachtas_members.csv", index=False)
print(f"\nSaved {len(df_members)} members")
print(df_members.head(10).to_string(index=False))

Fetched 100/1928 members
Fetched 200/1928 members
Fetched 300/1928 members
Fetched 400/1928 members
Fetched 500/1928 members
Fetched 600/1928 members
Fetched 700/1928 members
Fetched 800/1928 members
Fetched 900/1928 members
Fetched 1000/1928 members
Fetched 1100/1928 members
Fetched 1200/1928 members
Fetched 1300/1928 members
Fetched 1400/1928 members
Fetched 1500/1928 members
Fetched 1600/1928 members
Fetched 1700/1928 members
Fetched 1800/1928 members
Fetched 1900/1928 members
Fetched 1928/1928 members

Saved 2000 members
                  member_code       full_name_en  first_name last_name        party         constituency house_no                                                                              uri
Henry-J-J-Abbott.D.1987-03-10 Henry J. J. Abbott Henry J. J.    Abbott  Fianna Fáil   Longford-Westmeath       25 https://data.oireachtas.ie/ie/oireachtas/member/id/Henry-J-J-Abbott.D.1987-03-10
Caroline-Acheson.D.1981-06-30   Caroline Acheson    Caroline   Acheson  Fianna 

In [16]:
import pandas as pd

base = "/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/"

per = pd.read_csv(base + "per_entities.csv")
members = pd.read_csv(base + "oireachtas_members.csv")

# Build lookup sets from Oireachtas data
full_names = set(members["full_name_en"].str.lower().dropna())
last_names = set(members["last_name"].str.lower().dropna())
first_names = set(members["first_name"].str.lower().dropna())

results = []
for _, row in per.iterrows():
    entity = row["entity"]
    entity_lower = entity.lower().strip()
    
    # Check full name match
    if entity_lower in full_names:
        match_type = "full_name"
    # Check last name only match
    elif entity_lower in last_names:
        match_type = "last_name"
    # Check if entity contains a known last name
    elif any(ln in entity_lower for ln in last_names if len(ln) > 3):
        match_type = "partial"
    else:
        match_type = "no_match"
    
    results.append({
        "entity": entity,
        "train_count": row["train_count"],
        "in_test": row["in_test"],
        "match_type": match_type
    })

df = pd.DataFrame(results)
df.to_csv(base + "per_oireachtas_coverage.csv", index=False)

print("=== PER COVERAGE VIA OIREACHTAS ===")
for match_type in ["full_name", "last_name", "partial", "no_match"]:
    subset = df[df["match_type"] == match_type]
    print(f"\n{match_type}: {len(subset)}/{len(df)} ({100*len(subset)//len(df)}%)")
    print("  Examples:", subset["entity"].head(8).tolist())

print("\n=== UNMATCHED PER ENTITIES (sample) ===")
unmatched = df[df["match_type"] == "no_match"]
print(f"Total unmatched: {len(unmatched)}")
print(unmatched[["entity", "train_count", "in_test"]].head(20).to_string(index=False))

=== PER COVERAGE VIA OIREACHTAS ===

full_name: 1/651 (0%)
  Examples: ['Bertie Ahern']

last_name: 3/651 (0%)
  Examples: ['Andrews', 'Anthony', 'Berry']

partial: 4/651 (0%)
  Examples: ['Judith Cambell', 'Francis Bellew', 'Barry Keoghan', 'Máire Andrews']

no_match: 643/651 (98%)
  Examples: ['Aire Stáit', 'Taoiseach', 'Aire', 'tAire', 'Humphreys', 'Éamon Ó Cuív', 'Ó Cuív', 'Butler']

=== UNMATCHED PER ENTITIES (sample) ===
Total unmatched: 643
                  entity  train_count  in_test
              Aire Stáit           10     True
               Taoiseach            6     True
                    Aire            5    False
                   tAire            5    False
               Humphreys            5    False
            Éamon Ó Cuív            4    False
                  Ó Cuív            4     True
                  Butler            4    False
   Leas-Cheann Comhairle            3    False
             tAire Stáit            3    False
                    Ring       

In [17]:
# Check how Irish-language names are stored in Oireachtas data
irish_names = members[members["last_name"].str.contains("Ó |Ní |Mac |Mc |O'", na=False)]
print(f"Members with Irish-pattern names: {len(irish_names)}")
print(irish_names[["full_name_en", "last_name", "party"]].head(20).to_string(index=False))

Members with Irish-pattern names: 0
Empty DataFrame
Columns: [full_name_en, last_name, party]
Index: []


In [18]:
# Search for known Irish politicians by approximate name
test_names = ["Cuiv", "Snodaigh", "Humphreys", "Tubridy", "Dochartaigh"]

for name in test_names:
    matches = members[members["last_name"].str.contains(name, case=False, na=False)]
    if not matches.empty:
        print(f"\n'{name}' found:")
        print(matches[["full_name_en", "last_name", "party", "constituency"]].to_string(index=False))
    else:
        print(f"\n'{name}' — not found")


'Cuiv' — not found

'Snodaigh' — not found

'Humphreys' — not found

'Tubridy' — not found

'Dochartaigh' — not found


In [19]:
# Diagnose what we actually downloaded
print(f"Total rows: {len(members)}")
print(f"Unique member codes: {members['member_code'].nunique()}")
print(f"Unique full names: {members['full_name_en'].nunique()}")
print(f"\nHouse distribution:")
print(members["member_code"].str.extract(r'\.(D|S)\.')[0].value_counts())
print(f"\nSample of names containing 'O':")
o_names = members[members["last_name"].str.startswith("O", na=False)]
print(o_names[["full_name_en", "last_name"]].head(20).to_string(index=False))

print(f"\nAny Heather?")
print(members[members["first_name"].str.contains("Heather", case=False, na=False)][["full_name_en", "party", "constituency"]].to_string(index=False))

Total rows: 2000
Unique member codes: 100
Unique full names: 99

House distribution:
0
D    1460
S     540
Name: count, dtype: int64

Sample of names containing 'O':
Empty DataFrame
Columns: [full_name_en, last_name]
Index: []

Any Heather?
Empty DataFrame
Columns: [full_name_en, party, constituency]
Index: []


In [20]:
import requests
import pandas as pd
import time

HEADERS = {"User-Agent": "MichaelMarkeyDissertation/1.0 (TU Dublin MSc; michael.markey@tudublin.ie)"}
base = "/Users/michaelmarkey/Desktop/Dissertation - 27 05 2026/Train:Test:Val Split/"

all_members = []
limit = 100
offset = 0

while True:
    r = requests.get(
        "https://api.oireachtas.ie/v1/members",
        params={"limit": limit, "skip": offset, "format": "json"},
        headers=HEADERS,
        timeout=15
    )
    data = r.json()
    batch = data["results"]
    
    if not batch:
        break
        
    for item in batch:
        m = item["member"]
        party, constituency, house_no = None, None, None
        if m["memberships"]:
            latest = m["memberships"][0]["membership"]
            house_no = latest["house"]["houseNo"]
            if latest["parties"]:
                party = latest["parties"][0]["party"]["showAs"]
            if latest["represents"]:
                constituency = latest["represents"][0]["represent"]["showAs"]
        all_members.append({
            "member_code": m["memberCode"],
            "full_name_en": m["fullName"],
            "first_name": m["firstName"],
            "last_name": m["lastName"],
            "party": party,
            "constituency": constituency,
            "house_no": house_no,
            "uri": m["uri"]
        })
    
    offset += limit
    print(f"Fetched {offset} members so far (batch size: {len(batch)})")
    
    # Stop if we got fewer results than requested
    if len(batch) < limit:
        break
    
    time.sleep(0.3)

df_members = pd.DataFrame(all_members)
df_members.to_csv(base + "oireachtas_members.csv", index=False)
print(f"\nSaved {len(df_members)} rows")
print(f"Unique member codes: {df_members['member_code'].nunique()}")
print(f"Unique full names: {df_members['full_name_en'].nunique()}")
print(f"\nHeather Humphreys in dataset:")
print(df_members[df_members["last_name"].str.contains("Humphreys", case=False, na=False)][["full_name_en", "party", "constituency"]])

Fetched 100 members so far (batch size: 100)
Fetched 200 members so far (batch size: 100)
Fetched 300 members so far (batch size: 100)
Fetched 400 members so far (batch size: 100)
Fetched 500 members so far (batch size: 100)
Fetched 600 members so far (batch size: 100)
Fetched 700 members so far (batch size: 100)
Fetched 800 members so far (batch size: 100)
Fetched 900 members so far (batch size: 100)
Fetched 1000 members so far (batch size: 100)
Fetched 1100 members so far (batch size: 100)
Fetched 1200 members so far (batch size: 100)
Fetched 1300 members so far (batch size: 100)
Fetched 1400 members so far (batch size: 100)
Fetched 1500 members so far (batch size: 100)
Fetched 1600 members so far (batch size: 100)
Fetched 1700 members so far (batch size: 100)
Fetched 1800 members so far (batch size: 100)
Fetched 1900 members so far (batch size: 100)
Fetched 2000 members so far (batch size: 28)

Saved 1928 rows
Unique member codes: 1928
Unique full names: 1886

Heather Humphreys in d

In [21]:
!pip show pykeen

Name: pykeen
Version: 1.11.1
Summary: A package for training and evaluating multimodal knowledge graph embeddings
Home-page: https://github.com/pykeen/pykeen
Author: 
Author-email: Mehdi Ali <pykeen2019@gmail.com>, Max Berrendorf <max.berrendorf@gmail.com>, Laurent Vermue <pykeen2019@gmail.com>, Charles Tapley Hoyt <cthoyt@gmail.com>
License: MIT License

Copyright (c) 2019-2024 PyKEEN Project Team

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", W